##### Create a round robin matches for group of 3 teams

In [0]:
%sql
With teams as 
(
    Select 'India' as team_name
    union all
    Select 'Australia' 
    union all
    Select 'England'
),
pairs as 
(
    select t1.team_name as team1, t2.team_name as team2
    from teams t1
    join teams t2
      on t1.team_name < t2.team_name
),
rounds as
(
    select 1 as cycle_number
    union all  
    select 2
    union all
    select 3
)

select cycle_number, team1,team2,
row_number() over (order by cycle_number,team1,team2 ) as round_number
from pairs
cross join rounds
on 1=1
order by cycle_number

cycle_number,team1,team2,round_number
1,Australia,England,1
1,Australia,India,2
1,England,India,3
2,Australia,England,4
2,Australia,India,5
2,England,India,6
3,Australia,England,7
3,Australia,India,8
3,England,India,9


#### Employee Shift Rotation

You have 10 employees and 3 shifts (Morning, Afternoon, Night). Write a query to assign employees to shifts in a rotating fashion across 7 days.  

In [0]:
%sql
WITH employees AS (
    SELECT 'Alice' AS employee_name UNION ALL 
    SELECT 'Bob' UNION ALL
    SELECT 'Charlie' UNION ALL
    SELECT 'David' UNION ALL
    SELECT 'Eve' UNION ALL
    SELECT 'Frank' UNION ALL
    SELECT 'Grace' UNION ALL
    SELECT 'Heidi' UNION ALL
    SELECT 'Ivan' UNION ALL
    SELECT 'Judy'
),
employees_numbered AS (
    SELECT employee_name,
           ROW_NUMBER() OVER (ORDER BY employee_name) AS emp_no
    FROM employees
),
shifts AS (
    SELECT 0 AS shift_id, 'Morning' AS shift_name UNION ALL
    SELECT 1, 'Afternoon' UNION ALL
    SELECT 2, 'Evening'
),
working_days AS (
    SELECT 1 AS working_day_no, 'Monday' AS working_day UNION ALL
    SELECT 2, 'Tuesday' UNION ALL
    SELECT 3, 'Wednesday' UNION ALL
    SELECT 4, 'Thursday' UNION ALL
    SELECT 5, 'Friday' UNION ALL
    SELECT 6, 'Saturday' UNION ALL
    SELECT 7, 'Sunday'
)
SELECT 
    e.employee_name, 
    s.shift_name, 
    d.working_day
FROM employees_numbered e
CROSS JOIN working_days d
JOIN shifts s
  ON ((e.emp_no + d.working_day_no - 1) % 3) = s.shift_id
ORDER BY d.working_day_no, e.employee_name;


employee_name,shift_name,working_day
Alice,Afternoon,Monday
Bob,Evening,Monday
Charlie,Morning,Monday
David,Afternoon,Monday
Eve,Evening,Monday
Frank,Morning,Monday
Grace,Afternoon,Monday
Heidi,Evening,Monday
Ivan,Morning,Monday
Judy,Afternoon,Monday


In [0]:
# The list of column names
columns = ['emp_id', 'employee_name', 'department', 'salary', 'joining_date']

# The sample data structured as a list of rows (lists)
data = [
    [101, 'Alice Smith', 'HR', 60000, '2023-01-15'],
    [102, 'Bob Jones', 'Engineering', 95000, '2022-05-12'],
    [103, 'Charlie Brown', 'Finance', 75000, '2024-03-01'],
    [104, 'Diana Prince', 'Engineering', 110000, '2021-08-19'],
    [105, 'Evan Wright', 'Marketing', 68000, '2023-11-10'],
    [106, 'Fiona Gallagher', 'HR', 62000, '2025-02-01']
]

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession

df=spark.createDataFrame(data,columns)

df.createOrReplaceTempView("employee")

##### Given an array of strings, group anagrams together.

An anagram is a word formed by rearranging the letters of another word. For example, "eat", "tea", and "ate" are all anagrams of each other because they all use exactly one 'a', one 'e', and one 't'.

In [0]:
from collections import defaultdict

def groupAnagrams(strs):
    # Create a dictionary where the default value is an empty list
    anagram_map = defaultdict(list)
    
    for word in strs:
        # Sort the letters of the word to create a unique key
        # e.g., 'tea' -> ['e', 'a', 't'] -> sorted -> ['a', 'e', 't'] -> 'aet'
        sorted_key = "".join(sorted(word))
        
        # Append the original word to the list matching that key
        anagram_map[sorted_key].append(word)
        
    # Return just the grouped lists
    return list(anagram_map.values())

# Test it out
words = ["eat", "tea", "tan", "ate", "nat", "bat"]
print(groupAnagrams(words))


[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


##### Write a Python function to find the maximum repeating character in a string and its count, without using built-in libraries like Collections.

In [0]:
def max_repeating_char(s: str):
    if not s:
        return "", 0

    max_char = s[0]
    max_count = 1
    
    current_char = s[0]
    current_count = 1

    for i in range(1, len(s)):
        if s[i] == current_char:
            current_count += 1
        else:
            if current_count > max_count:
                max_count = current_count
                max_char = current_char
            current_char = s[i]
            current_count = 1
            
    # Final check for the last sequence in the string
    if current_count > max_count:
        max_count = current_count
        max_char = current_char

    return max_char, max_count

# Example Test
print(max_repeating_char("aaabbdddddcccccddddddd")) 

('d', 7)


##### Processing a Massive File via Generator (yield)
The Goal: Stream a file line by line without pulling the whole file into system memory.

In [0]:
def read_large_file(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            yield line.strip()

# Databricks compute runs in the cloud and cannot see your local Windows C:\ drive,
# so a local path like C:\Users\dilip\massive_50gb_data.txt raises FileNotFoundError.
# Upload the file to Databricks first (e.g. via Catalog > Create Volume, then upload
# into the volume, or with the CLI: databricks fs cp massive_50gb_data.txt dbfs:/FileStore/),
# then reference the cloud path here.

# Usage in a processing pipeline
for clean_line in read_large_file('/Volumes/workspace/default/landing_volume/massive_50gb_data.txt'):
    # Process your line here without crashing the 8GB RAM machine
    if "ERROR" in clean_line:
        print(clean_line)

ERROR  : go check your file
ERROR: Name missing in the data


#####  3rd Highest Salary in Each Department

The Goal: Handle duplicate high salaries correctly using DENSE_RANK().

In [0]:
columns = ["employee_id", "employee_name", "department", "salary", "manager_id"]

data = [
    (101, "John Smith",      "IT",      135000, 201),
    (102, "Alice Johnson",   "IT",      85000, 201),
    (103, "Bob Williams",    "HR",      70000, 202),
    (104, "Carol Brown",     "HR",      75000, 202),
    (105, "David Miller",    "Finance", 90000, 203),
    (106, "Emma Davis",      "Finance", 80000, 203),
    (107, "Frank Wilson",    "IT",      85000, 201),
    (108, "Grace Taylor",    "Sales",   120000, 204),
    (109, "Henry Moore",     "Sales",   68000, 204),
    (110, "Ivy Anderson",    "Marketing",76000, 205),
    (201, "Michael Scott",   "IT",     130000, 301),
    (202, "Sarah Connor",    "HR",     120000, 301),
    (203, "Robert King",     "Finance",145000, 301),
    (204, "Nancy White",     "Sales",  118000, 301),
    (205, "James Clark",     "Marketing",122000, 301),
    (301, "Dilip M G",     "CEO",140000, None),
     (302, "Emma Davis Jr",      "Finance", 80000, 203)
]

df= spark.createDataFrame(data,columns)
df.createOrReplaceTempView('employees')

In [0]:
%sql
WITH RankedSalaries AS (
    SELECT 
        employee_id,
        employee_name,
        department,
        salary,
        DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) as rnk
    FROM employees
)
SELECT employee_id,employee_name, department, salary
FROM RankedSalaries
WHERE rnk = 3;

employee_id,employee_name,department,salary
106,Emma Davis,Finance,80000
302,Emma Davis Jr,Finance,80000
103,Bob Williams,HR,70000
102,Alice Johnson,IT,85000
107,Frank Wilson,IT,85000
109,Henry Moore,Sales,68000


##### Delete Exact Duplicate Records (No Primary Key)
The Goal: Keep only one instance of completely identical rows using a CTE.